<a href="https://www.kaggle.com/code/devanshshukla123/analogy-nlp?scriptVersionId=284031025" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/glove6b100dtxt/glove.6B.100d.txt


In [2]:
import numpy as np
from sklearn.metrics.pairwise import pairwise_distances


# using the dataset to classify a word2vec dictionary with words as key and value as its correspnding vector

In [3]:
word2vec = {}
embedding = []
idx2word = []
with open('/kaggle/input/glove6b100dtxt/glove.6B.100d.txt') as f:
    for row in f:
        value = row.split(" ")
        word = value[0]
        vec = np.asarray(value[1:], dtype = 'float32')
        word2vec[word] = vec
        embedding.append(vec)
        idx2word.append(word)

print(f'Found {len(word2vec)} word vectors')
embedding = np.array(embedding)
V,D = embedding.shape
print(embedding.shape)

Found 400000 word vectors
(400000, 100)


# checking the best possible 4 words that are closer to y1 in the same sense of relation as in x1 and x2

In [4]:
def analogy(x1, x2, y1):
    for w in (x1, x2, y1):
        if w not in word2vec:
            print(f"{w} not in dictionary")
            return
    x1vec = word2vec[x1]
    x2vec = word2vec[x2]
    y1vec = word2vec[y1]
    Vo = x2vec - x1vec + y1vec
    
    distance = pairwise_distances(Vo.reshape(1,D), embedding, metric = 'l2').reshape(V)
    ids = distance.argsort()[:4]
    words = [idx2word[idm] for idm in ids]
     
    best = [word for word in words if word not in (x1, x2, y1)]
    print('best match word ', best)
    print(f'so, {x1} - {x2} = {y1} - {best[0]}')

In [5]:
analogy('king','man','queen')

best match word  ['woman', 'girl', 'she']
so, king - man = queen - woman


In [6]:
analogy('electron','proton','negative')


best match word  ['positive', 'impression', 'bullish']
so, electron - proton = negative - positive


In [7]:
analogy('india','cricket','usa')

best match word  ['rugby', 'basketball', 'football']
so, india - cricket = usa - rugby


In [8]:
analogy('maths','algebra','football')

best match word  ['team', 'soccer', 'club']
so, maths - algebra = football - team


![Word vector similarity](https://miro.medium.com/1*LdviucnshWgIIcQvhTTF-g.png)

**Words are mapped to vectors where the vector difference between "king" and "man" is similar to "queen" and "woman" (e.g., vector('king') - vector('man') ≈ vector('queen') - vector('woman'))**

**similarity being calculated on the basis of Euclidean Distance: Measures the straight-line distance between vectors; smaller distance means closer words.** 